# Phase 1: Data Preparation & Integrity

**Goal:** Load the FMCG sales dataset, validate its structure, and identify data quality
issues (missing values, duplicates, invalid values, inconsistent categories) before analysis.

**Steps:** load data → check date range → duplicates and categories → dataset size →
series completeness → invalid values → zero-sales check → save cleaned data

In [1]:
import pandas as pd

df = pd.read_csv("../data/fmcg_sales_3years_1M_rows.csv")

print(df.shape)
df.info()
df.head()

(1100000, 33)
<class 'pandas.DataFrame'>
RangeIndex: 1100000 entries, 0 to 1099999
Data columns (total 33 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   date            1100000 non-null  str    
 1   year            1100000 non-null  int64  
 2   month           1100000 non-null  int64  
 3   day             1100000 non-null  int64  
 4   weekofyear      1100000 non-null  int64  
 5   weekday         1100000 non-null  int64  
 6   is_weekend      1100000 non-null  int64  
 7   is_holiday      1100000 non-null  int64  
 8   temperature     1100000 non-null  float64
 9   rain_mm         1100000 non-null  float64
 10  store_id        1100000 non-null  str    
 11  country         1100000 non-null  str    
 12  city            1100000 non-null  str    
 13  channel         1100000 non-null  str    
 14  latitude        1100000 non-null  float64
 15  longitude       1100000 non-null  float64
 16  sku_id          1100000 non-null 

,date,year,month,day,weekofyear,weekday,is_weekend,is_holiday,temperature,rain_mm,...,discount_pct,promo_flag,gross_sales,net_sales,stock_on_hand,stock_out_flag,lead_time_days,supplier_id,purchase_cost,margin_pct
0,2021-01-01,2021,1,1,53,4,0,1,8.44,1.24,...,0.1,1,167.84,151.06,248,0,11,S008,7.53,0.182
1,2021-01-02,2021,1,2,53,5,1,0,12.61,1.12,...,0.0,0,125.88,125.88,238,0,6,S057,5.19,0.505
2,2021-01-03,2021,1,3,53,6,1,0,12.02,2.69,...,0.3,1,398.62,279.03,238,0,6,S017,5.59,0.168
3,2021-01-04,2021,1,4,1,0,0,0,7.76,4.65,...,0.0,0,83.92,83.92,216,0,7,S012,7.81,0.255
4,2021-01-05,2021,1,5,1,1,0,0,11.16,1.77,...,0.2,1,178.33,142.66,372,0,8,S038,7.62,0.073


In [4]:
# Convert text dates to datetime so we can sort, filter and build lags by time later
df["date"] = pd.to_datetime(df["date"])

print("Date range:", df["date"].min(), "to", df["date"].max())
print("Unique days:", df["date"].nunique())
# Finding: dates run 2021-01-01 to 2023-12-31, 1,095 unique days, no missing calendar days

Date range: 2021-01-01 00:00:00 to 2023-12-31 00:00:00
Unique days: 1095


In [5]:
# Exact duplicate rows (every column identical)
print("Duplicate rows:", df.duplicated().sum())

# Duplicates on the natural key: one record per day, store and SKU is expected
key = ["date", "store_id", "sku_id"]
print("Duplicate date-store-SKU records:", df.duplicated(subset=key).sum())

# Categorical columns: check for inconsistent spellings (e.g. 'Supermarket' vs 'supermarket')
for col in ["country", "channel", "category"]:
    print(f"\n{col}:", df[col].nunique(), "unique values")
    print(df[col].value_counts())

Duplicate rows: 0
Duplicate date-store-SKU records: 0

country: 7 unique values
country
Italy          350400
Spain          262800
Germany        175200
Poland          87600
France          87600
Austria         87600
Netherlands     48800
Name: count, dtype: int64

channel: 4 unique values
channel
Hypermarket    525600
Supermarket    262800
E-commerce     262800
Convenience     48800
Name: count, dtype: int64

category: 5 unique values
category
Beverages        266085
Snacks           261705
Personal Care    199290
Dairy            196005
Home Care        176915
Name: count, dtype: int64


In [9]:
print("Countries:", df["country"].nunique())
print("Cities:   ", df["city"].nunique())
print("Stores:   ", df["store_id"].nunique())
print("SKUs:     ", df["sku_id"].nunique())
print("Suppliers:", df["supplier_id"].nunique())

Countries: 7
Cities:    9
Stores:    13
SKUs:      102
Suppliers: 60


**Finding:** The dataset contains 7 countries, 9 cities, 13 stores, 102 SKUs and 60 suppliers.
This is smaller than the original project brief (15 countries, 350+ SKUs), so all analysis
below uses the actual data.


In [10]:
series_len = df.groupby(["store_id", "sku_id"]).size()
print("Store-SKU series:", len(series_len))
series_len.describe()

Store-SKU series: 1005


count    1005.000000
mean     1094.527363
std        14.983407
min       620.000000
25%      1095.000000
50%      1095.000000
75%      1095.000000
max      1095.000000
dtype: float64

**Finding:** There are 1,005 store-SKU series. Most (over 75%) have a complete 1,095 days;
the shortest has 620 days, likely SKUs introduced or discontinued part-way through.
Series are complete enough for daily lag features at store-SKU level.

In [11]:
checks = {
    "units_sold <= 0":          (df["units_sold"] <= 0).sum(),
    "lead_time_days <= 0":      (df["lead_time_days"] <= 0).sum(),
    "discount_pct outside 0-1": ((df["discount_pct"] < 0) | (df["discount_pct"] > 1)).sum(),
    "net_sales > gross_sales":  (df["net_sales"] > df["gross_sales"]).sum(),
    "stock_on_hand < 0":        (df["stock_on_hand"] < 0).sum(),
}
pd.Series(checks, name="rows affected")

units_sold <= 0             3092
lead_time_days <= 0            0
discount_pct outside 0-1       0
net_sales > gross_sales        0
stock_on_hand < 0              0
Name: rows affected, dtype: int64

In [12]:
bad = df[df["units_sold"] <= 0]

# Are they zeros or negatives?
print("Zero sales:    ", (bad["units_sold"] == 0).sum())
print("Negative sales:", (bad["units_sold"] < 0).sum())

# Do they coincide with stock-outs?
print("\nStock-out flag among these rows:")
print(bad["stock_out_flag"].value_counts())

Zero sales:     3092
Negative sales: 0

Stock-out flag among these rows:
stock_out_flag
1    2534
0     558
Name: count, dtype: int64


**Finding:** 3,092 rows (0.3%) have `units_sold = 0`; none are negative. About 82% coincide with
a stock-out, so these reflect unavailable stock rather than absent demand (censored demand).
The remaining 18% are genuine zero-demand days. All rows are **kept**; `stock_out_flag` will be
used as a feature in Phase 4.

In [13]:
# Save the checked dataset (date converted to datetime). This stays local because of .gitignore.
df.to_csv("../data/fmcg_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (1100000, 33)


## Phase 1 findings

- **Structure:** 1,100,000 rows × 33 columns covering 2021-01-01 to 2023-12-31 (1,095 days, no gaps).
  7 countries, 9 cities, 13 stores, 102 SKUs, 60 suppliers (smaller than the original brief).
- **Missing values:** none in any column.
- **Duplicates:** none, either as exact rows or on the date-store-SKU key.
- **Categories:** channel, country and category labels were already consistent, so no standardization was needed.
- **Lead time:** no zero or negative values, so no imputation was needed.
- **Zero sales:** 3,092 rows (0.3%) have `units_sold = 0`, mostly during stock-outs (censored demand). All kept;
  `stock_out_flag` will be used as a feature in Phase 4.
- **Series completeness:** 1,005 store-SKU series, most with the full 1,095 days.
- **Output:** cleaned dataset saved locally as `data/fmcg_clean.csv` (excluded from Git for size).